In [3]:
import sys
import os

# Check for a modern PEP 405 virtual environment
is_venv = sys.prefix != sys.base_prefix

print(f"Is inside a venv? -> {is_venv}")
print(f"Current Environment Path: {sys.prefix}")
print(f"Active Python Executable: {sys.executable}")

# Optional check: If you activated it via terminal scripts
if 'VIRTUAL_ENV' in os.environ:
    print(f"Activated via terminal VIRTUAL_ENV: {os.environ['VIRTUAL_ENV']}")


Is inside a venv? -> True
Current Environment Path: c:\Users\arz_2\Basira_Motassim_Group_A\.venv
Active Python Executable: c:\Users\arz_2\Basira_Motassim_Group_A\.venv\Scripts\python.exe
Activated via terminal VIRTUAL_ENV: C:\Users\arz_2\Basira_Motassim_Group_A\.venv


In [ ]:
import glob
import os
import pandas as pd

In [10]:
import glob

files = glob.glob("../mutation/**/*.maf.gz", recursive=True)
if not files:
    raise SystemExit("No .maf.gz files found under mutation/")

print(f"Found {len(files)} files")

Found 434 files


In [11]:
maf = pd.concat(
    [pd.read_csv(f, sep="\t", comment="#", low_memory=False,
                 usecols=["Hugo_Symbol", "Variant_Classification",
                          "Tumor_Sample_Barcode"])
     for f in files],
    ignore_index=True,
)

In [12]:
# ---- 2. IDs from the barcode (TCGA-XX-XXXX-01A-...) --------------------------
maf["patient"] = maf["Tumor_Sample_Barcode"].str[:12]
maf["sample_type"] = maf["Tumor_Sample_Barcode"].str[13:15]
maf["sample"] = maf["Tumor_Sample_Barcode"].str[:16]  # patient + sample type + vial

In [17]:
maf.head()

,Hugo_Symbol,Variant_Classification,Tumor_Sample_Barcode,patient,sample_type,sample
0,RHBG,Missense_Mutation,TCGA-BR-8678-01A-11D-2394-08,TCGA-BR-8678,01,TCGA-BR-8678-01A
1,FCRL6,Missense_Mutation,TCGA-BR-8678-01A-11D-2394-08,TCGA-BR-8678,01,TCGA-BR-8678-01A
2,BRINP3,Silent,TCGA-BR-8678-01A-11D-2394-08,TCGA-BR-8678,01,TCGA-BR-8678-01A
3,RAB3GAP2,Silent,TCGA-BR-8678-01A-11D-2394-08,TCGA-BR-8678,01,TCGA-BR-8678-01A
4,PRSS38,Missense_Mutation,TCGA-BR-8678-01A-11D-2394-08,TCGA-BR-8678,01,TCGA-BR-8678-01A


In [18]:
maf["sample_type"].value_counts()

sample_type
01    183107
Name: count, dtype: int64

In [23]:
# Distribution: how many patients have 1 sample, how many have 2, etc.
samples_per_patient.value_counts().sort_index()


sample
1    431
Name: count, dtype: int64

## analysis
the data contains primary tumors and all samples are unique (one sample per patient)

In [31]:
patients = set(maf["patient"])
n_patients = len(patients)

In [32]:
n_patients

431

# Gene list
CDH1, CDH11, CDH13, CDH17, CDH2, CDH3, CDHR2, CDHR5, CLSTN1, DSC2, DSG2, FAT1, PCDH1, PCDH17, PCDH18, PCDHGA10

*Paper*: Wang H, Mazzocca A, Gao P. “Cadherin dysregulation in gastric cancer: insights into gene expression, pathways, and prognosis.” J Gastrointest Oncol 2023;14(5):2064-2082.
*Aim*: A bioinformatics survey of the whole cadherin family in stomach adenocarcinoma (STAD). It looks at expression, mutations, survival, microRNAs targeting cadherins, and immune-cell infiltration. It uses public data only (TCGA, GTEx, cBioPortal, GEPIA2), with no new patients or experiments.

### How they got the 16 genes
- Build the family list. They took all 125 cadherin-family genes from the HGNC gene-group portal. These span seven groups: clustered protocadherins (64), cadherin-related (17), type II classical (13), non-clustered protocadherins (12), desmosomal (7), type I classical (5), CELSR (3), major cadherins (2) and 7D cadherins (2).
- Test each gene for differential expression. They compared STAD tumours (TCGA, n=408) with normal tissue (TCGA plus GTEx, n=211) using GEPIA2, which uses limma.
- Apply a cut-off. A gene was kept if |log2 fold change| was greater than 1 and the Benjamini-Hochberg adjusted P was below 0.05. That left 16 of 125. All were upregulated in tumours except CDH2, which was downregulated.
So the 16 are the genes that differ between tumour and normal. They were not selected for anything to do with TP53, immunity or survival.

### Other findings
Alterations: Mutations or copy-number changes in at least one of the 16 appeared in 227 patients (about 52%). PCDH17 (12.3%) and FAT1 (about 12%) were the most often altered, and PCDHGA10 the least (2.7%).
Effects of mutation: Mutations in five genes (CDH1, CDHR2, FAT1, PCDH17, PCDHGA10) changed the expression of other genes. Mutations in the other genes mostly lowered it.
Survival: Seven genes were significantly linked to overall survival. Lower expression went with better survival for six of them (CDH2, CDH11, CDH13, PCDH17, PCDH18, PCDHGA10). FAT1 went the other way.
Immune cells: The abundance of 11 of 24 immune cell types differed between tumour and normal. CDH1 and CDH17 correlated with all five of the most abundant types (B, DC, NK, CD4 T and NKT cells), mostly negatively.

### Important points
- The immune part: They correlated each gene’s expression with immune cells. They did not compare groups defined by mutation. Their only mutation analysis asked which other genes change expression in cadherin-mutant tumours.
- They did not look at TP53 or at co-mutation. I found no TP53 analysis in the pages I read.
- There is no CD8 or cytolytic readout. CD8 naive cells appear in the correlation grid, but CD8+ T cells and cytolytic activity weren’t reported as an outcome.

### Limits 
- It is a differential-expression list, not a ranking of causal importance. “Identified as important” in your earlier prompt overstates it. These genes are simply the ones that differ most in tumours.
- The tumour-normal comparison mixes sources. Normal samples come from TCGA plus GTEx, which can add batch effects.
- The authors flag a histology issue. The TCGA sample pool is 41% intestinal and only 16% diffuse, which they say could explain the CDH1 upregulation.
- The miRNA links are unverified. Only one of the 23 miRNA-cadherin pairs is experimentally confirmed (hsa-miR-548ba with PCDH17). The authors say the rest need validation.

## Table 1. Cadherins differentially expressed between stomach adenocarcinoma and normal tissue samples.

![Table 1. Cadherins differentially expressed between stomach adenocarcinoma and normal tissue samples.
](../assets/gene_list_table.jpeg)


In [34]:
# ---- 5. Non-silent variants (no Nonstop_Mutation, as in the plan) -------------
keep = ["Missense_Mutation", "Nonsense_Mutation", "Frame_Shift_Del",
        "Frame_Shift_Ins", "In_Frame_Del", "In_Frame_Ins", "Splice_Site",
        "Translation_Start_Site"]
ns = maf[maf["Variant_Classification"].isin(keep)]

In [37]:
# ---- 6. Gene sets: TP53, and the cadherin panel from the DE table -------------
CAD_GENES = [
    "CDH1", "CDH11", "CDH13", "CDH17", "CDH2", "CDH3",
    "CDHR2", "CDHR5", "CLSTN1", "DSC2", "DSG2", "FAT1",
    "PCDH1", "PCDH17", "PCDH18", "PCDHGA10",
]

tp53 = set(ns.loc[ns["Hugo_Symbol"] == "TP53", "patient"])
cad  = set(ns.loc[ns["Hugo_Symbol"].isin(CAD_GENES), "patient"])

In [38]:
found = set(CAD_GENES) & set(ns["Hugo_Symbol"])
missing = set(CAD_GENES) - found
print("Found:", sorted(found))
print("Missing from MAF:", sorted(missing))

Found: ['CDH1', 'CDH11', 'CDH13', 'CDH17', 'CDH2', 'CDH3', 'CDHR2', 'CDHR5', 'CLSTN1', 'DSC2', 'DSG2', 'FAT1', 'PCDH1', 'PCDH17', 'PCDH18', 'PCDHGA10']
Missing from MAF: []


In [41]:
len(cad)

178

In [44]:
# ---- Cell: Four groups ----
cad_only  = cad - tp53
tp53_only = tp53 - cad
co_mut    = tp53 & cad
wild_type = patients - tp53 - cad

print(f"Cadherin only (no TP53):   {len(cad_only):>4}  ({100*len(cad_only)/n_patients:.1f}%)")
print(f"TP53 only (no cadherin):   {len(tp53_only):>4}  ({100*len(tp53_only)/n_patients:.1f}%)")
print(f"Co-mutated (TP53 + cad):   {len(co_mut):>4}  ({100*len(co_mut)/n_patients:.1f}%)")
print(f"Wild-type (neither):       {len(wild_type):>4}  ({100*len(wild_type)/n_patients:.1f}%)")
print(f"Total:                     {len(cad_only)+len(tp53_only)+len(co_mut)+len(wild_type):>4}")

print(f"TP53 cases percentage (regardless of other mutations):   {len(tp53):>4}  ({100*len(tp53)/n_patients:.1f}%)")


Cadherin only (no TP53):     96  (22.3%)
TP53 only (no cadherin):    115  (26.7%)
Co-mutated (TP53 + cad):     82  (19.0%)
Wild-type (neither):        138  (32.0%)
Total:                      431
TP53 cases percentage (regardless of other mutations):    197  (45.7%)


In [45]:
# ---- Cell: Patients per cadherin gene, split by TP53 status ----
gene_patients = {g: set(ns.loc[ns["Hugo_Symbol"] == g, "patient"]) for g in CAD_GENES}

for g in CAD_GENES:
    pts = gene_patients[g]
    if not pts:
        continue
    with_tp53 = len(pts & tp53)
    without_tp53 = len(pts) - with_tp53
    print(f"{g:<10} {len(pts):>4} patients  ({100*len(pts)/n_patients:>5.1f}%)  "
          f"| +TP53: {with_tp53:>3}  -TP53: {without_tp53:>3}")

CDH1         39 patients  (  9.0%)  | +TP53:   9  -TP53:  30
CDH11        34 patients  (  7.9%)  | +TP53:  22  -TP53:  12
CDH13        13 patients  (  3.0%)  | +TP53:   6  -TP53:   7
CDH17         8 patients  (  1.9%)  | +TP53:   4  -TP53:   4
CDH2         20 patients  (  4.6%)  | +TP53:  11  -TP53:   9
CDH3         11 patients  (  2.6%)  | +TP53:   8  -TP53:   3
CDHR2        21 patients  (  4.9%)  | +TP53:   5  -TP53:  16
CDHR5         8 patients  (  1.9%)  | +TP53:   1  -TP53:   7
CLSTN1       12 patients  (  2.8%)  | +TP53:   5  -TP53:   7
DSC2         12 patients  (  2.8%)  | +TP53:   2  -TP53:  10
DSG2         10 patients  (  2.3%)  | +TP53:   6  -TP53:   4
FAT1         37 patients  (  8.6%)  | +TP53:  14  -TP53:  23
PCDH1        16 patients  (  3.7%)  | +TP53:   7  -TP53:   9
PCDH17       46 patients  ( 10.7%)  | +TP53:  27  -TP53:  19
PCDH18       24 patients  (  5.6%)  | +TP53:  10  -TP53:  14
PCDHGA10     11 patients  (  2.6%)  | +TP53:   2  -TP53:   9


In [46]:
# ---- Cell: Sanity checks ----
assert len(cad_only) + len(tp53_only) + len(co_mut) + len(wild_type) == n_patients
assert tp53 <= patients and cad <= patients
assert set().union(*gene_patients.values()) == cad
print("All checks passed.")

All checks passed.


In [ ]:
# ---- Cell: Summary ----
print(f"Total patients (primary tumours, one sample each): {n_patients}")